<a href="https://colab.research.google.com/github/lucasmsorrentino/FrameworksAI/blob/main/3_Resolu%C3%A7%C3%A3o_do_Exerc%C3%ADcio_de_Sistemas_de_Recomenda%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IAA012 - Frameworks de IA - Trabalho Final
## Integrantes: Lucas de Castro Oliveira e Lucas Martins Sorrentino

## 3. Resolução do Exercício de Sistemas de Recomendação

---

### 1. Setup do Ambiente

In [1]:
!pip install -q tensorflow-recommenders
import os
# OBRIGATÓRIO: Forçar Keras 2 antes de importar TensorFlow
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs

print(f"Versão TF: {tf.__version__}")


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


2026-03-08 16:05:34.915247: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Versão TF: 2.20.0


### 2. Carregamento e Correção dos dos Dados

In [2]:
# Procura o arquivo automaticamente
files = [f for f in os.listdir() if 'Base_livros' in f and not f.endswith('.csv')] # Prioriza Excel
if not files:
    # Se não achar Excel, tenta CSV
    files = [f for f in os.listdir() if 'Base_livros' in f]

if not files:
    raise FileNotFoundError("❌ ERRO: Arquivo 'Base_livros' não encontrado. Faça upload!")

filename = files[0]
print(f"-> Lendo: {filename}")

try:
    # Tenta ler o arquivo
    if filename.endswith('.xlsx'):
        df = pd.read_excel(filename)
    else:
        df = pd.read_csv(filename, on_bad_lines='skip')

    # --- DIAGNÓSTICO DE COLUNAS ---
    print(f"-> Colunas detetadas inicialmente: {df.columns.tolist()}")

    # --- CORREÇÃO DE FORMATO "AGLOMERADO" ---
    # Verifica se a primeira coluna contém vírgulas no nome (sinal de CSV mal formatado)
    primeira_coluna = df.columns[0]

    # Se 'ID_usuario' NÃO existe, mas a 1ª coluna parece conter os dados misturados...
    if 'ID_usuario' not in df.columns and ',' in str(primeira_coluna):
        print("⚠️ Detetado formato misturado (CSV dentro de Excel). A corrigir à força...")

        # Pega os nomes corretos do cabeçalho "ISBN,Titulo,Autor..."
        novos_nomes = str(primeira_coluna).replace('"', '').split(',')

        # Separa a primeira coluna pelos dados
        df_split = df.iloc[:, 0].astype(str).str.split(',', expand=True)

        # Garante que não temos colunas a mais
        if df_split.shape[1] > len(novos_nomes):
            df_split = df_split.iloc[:, :len(novos_nomes)]

        # Atribui os nomes
        df_split.columns = novos_nomes
        df = df_split
        print("✅ Correção aplicada com sucesso!")

    # Limpeza final de nomes (remove espaços extras)
    df.columns = df.columns.astype(str).str.strip()

    # --- VERIFICAÇÃO FINAL ---
    if 'ID_usuario' not in df.columns:
        print("❌ ERRO CRÍTICO: As colunas ainda estão erradas.")
        print(f"O Python está lendo isto: {df.columns.tolist()}")
        print("Tente converter o seu Excel para CSV 'verdadeiro' no Excel (Salvar Como > CSV UTF-8).")
        raise KeyError("Coluna ID_usuario não encontrada")

    # Limpeza dos valores
    df['ID_usuario'] = df['ID_usuario'].astype(str).str.replace('"', '').str.strip()
    df['Titulo'] = df['Titulo'].astype(str).str.replace('"', '').str.strip()

    print("✅ Dados prontos!")

except Exception as e:
    print(f"❌ Ocorreu um erro na leitura: {e}")
    raise

-> Lendo: Base_livros.xlsx
-> Colunas detetadas inicialmente: ['ISBN,Titulo,Autor,Ano,Editora,ID_usuario,Notas', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6']
⚠️ Detetado formato misturado (CSV dentro de Excel). A corrigir à força...
✅ Correção aplicada com sucesso!
✅ Dados prontos!


### 3. Preparar Modelo

In [3]:
data = df[['ID_usuario', 'Titulo']].astype(str)
ratings_ds = tf.data.Dataset.from_tensor_slices({
    "ID_usuario": data['ID_usuario'].values.astype(str),
    "Titulo": data['Titulo'].values.astype(str)
})
books_ds = tf.data.Dataset.from_tensor_slices(data['Titulo'].unique().astype(str))

ratings_ds = ratings_ds.map(lambda x: {"user_id": x["ID_usuario"], "book_title": x["Titulo"]})

# Vocabulários
user_lookup = tf.keras.layers.StringLookup(mask_token=None)
user_lookup.adapt(ratings_ds.map(lambda x: x["user_id"]))

book_lookup = tf.keras.layers.StringLookup(mask_token=None)
book_lookup.adapt(books_ds)

num_users = user_lookup.vocabulary_size()
num_books = book_lookup.vocabulary_size()
print(f"-> {num_users} usuários e {num_books} livros.")

# Classes do Modelo (Customizadas para estabilidade)
class UserTower(tf.keras.Model):
    def __init__(self, lookup, vocab_size, emb_dim=32):
        super().__init__()
        self.lookup = lookup
        self.embedding = tf.keras.layers.Embedding(vocab_size, emb_dim)
    def call(self, inputs):
        return self.embedding(self.lookup(inputs))

class BookTower(tf.keras.Model):
    def __init__(self, lookup, vocab_size, emb_dim=32):
        super().__init__()
        self.lookup = lookup
        self.embedding = tf.keras.layers.Embedding(vocab_size, emb_dim)
    def call(self, inputs):
        return self.embedding(self.lookup(inputs))

class BookRecModel(tfrs.Model):
    def __init__(self):
        super().__init__()
        self.user_model = UserTower(user_lookup, num_users)
        self.book_model = BookTower(book_lookup, num_books)
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=books_ds.batch(128).map(self.book_model)
            )
        )
    def compute_loss(self, features, training=False):
        u = self.user_model(features["user_id"])
        b = self.book_model(features["book_title"])
        return self.task(u, b)

# Treinar
model = BookRecModel()
model.compile(optimizer=tf.keras.optimizers.Adagrad(0.1))
model.fit(ratings_ds.batch(4096).cache(), epochs=10)

I0000 00:00:1772996844.074741    6687 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2146 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


-> 13857 usuários e 115215 livros.
Epoch 1/10


2026-03-08 16:09:32.786343: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fa7064e2e80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-08 16:09:32.786360: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-03-08 16:09:32.792320: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-08 16:09:32.909346: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91701
I0000 00:00:1772996973.568546    7073 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


32/32 [==============================] - 369s 11s/step - factorized_top_k/top_1_categorical_accuracy: 7.7617e-06 - factorized_top_k/top_5_categorical_accuracy: 4.9675e-04 - factorized_top_k/top_10_categorical_accuracy: 7.5288e-04 - factorized_top_k/top_50_categorical_accuracy: 0.0014 - factorized_top_k/top_100_categorical_accuracy: 0.0018 - loss: 32853.9382 - regularization_loss: 0.0000e+00 - total_loss: 32853.9382
Epoch 2/10
32/32 [==============================] - 369s 12s/step - factorized_top_k/top_1_categorical_accuracy: 4.6570e-05 - factorized_top_k/top_5_categorical_accuracy: 0.0034 - factorized_top_k/top_10_categorical_accuracy: 0.0056 - factorized_top_k/top_50_categorical_accuracy: 0.0147 - factorized_top_k/top_100_categorical_accuracy: 0.0235 - loss: 32664.8746 - regularization_loss: 0.0000e+00 - total_loss: 32664.8746
Epoch 3/10
32/32 [==============================] - 368s 12s/step - factorized_top_k/top_1_categorical_accuracy: 4.4242e-04 - factorized_top_k/top_5_categorica

### 4. Resultados

In [4]:
print("=== RESULTADO ===")
index = tfrs.layers.factorized_top_k.BruteForce(model.user_model)
index.index_from_dataset(
  tf.data.Dataset.zip((books_ds.batch(100), books_ds.batch(100).map(model.book_model)))
)

test_user = '276725'
print(f"Top 3 sugestões para o usuário {test_user}:")
_, titles = index(np.array([test_user]))
for i, title in enumerate(titles[0, :3]):
    print(f"{i+1}: {title.numpy().decode('utf-8')}")

=== RESULTADO ===


2026-03-08 17:13:14.216997: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Top 3 sugestões para o usuário 276725:
1: Celebrity
2: Anne Frank's Tales from the Secret Annex
3: Gentlemen Prefer Blondes: The Illuminating Diary of a Professional Lady


### 8. Análise dos Resultados

1. Desempenho do Modelo O modelo foi treinado com sucesso numa base de dados
considerável (13.857 usuários e 115.215 livros).
    * Convergência: Observamos que a perda (loss) diminuiu progressivamente ao longo das 10 épocas (de 32853 para 22035), o que prova que a rede neural aprendeu padrões de associação entre leitores e livros.
    * Precisão: A métrica top_100_categorical_accuracy subiu para cerca de 48% (0.047860). Num universo de 115 mil livros possíveis, o modelo conseguir colocar o livro "correto" entre os 100 primeiros em 48% das vezes (após apenas 10 épocas de treino) demonstra que o sistema de Retrieval (Recuperação) está funcionando muito melhor que um sorteio aleatório.

2. As Recomendações para o Usuário 276725 o sistema sugeriu os seguintes livros:

    1.   Celebrity
    2.   Anne Frank's Tales from the Secret Annex.
    3.   Gentlemen Prefer Blondes: The Illuminating Diary of a Professional Lady.
